## Analysis notebook for the toy network results
loads estimated fluxes from the reference simulation, computes summary statistics, and formats a flux results table.

In [1]:
import pickle
from pathlib import Path

import jax.numpy as jnp
import numpy as np
from jax import config

config.update("jax_enable_x64", True)

import polars as pl

In [2]:
RESULTS_DIR = Path("..") / "data" / "results" / "toy_example"
PLOT_DIR = Path("..") / "figures"
true_flux = np.load(RESULTS_DIR / "true_flux.npy")
with open(RESULTS_DIR / "estimated_fluxes_reference.pkl", "rb") as f:
    estimated_fluxes_ref = pickle.load(f)

In [3]:
# Column order in the estimated_fluxes_ref array
col_order = ["v2", "v4", "v6", "v1", "v3", "v5"]
col_index = {name: i for i, name in enumerate(col_order)}

# true_flux array is ordered v1..v6
true_flux_map = {f"v{i+1}": float(true_flux[i]) for i in range(6)}

flux_keys = ["v2", "v3", "v4", "v5", "v6"]  # v1 excluded

rows = []
for model in ("trad", "ilr"):
    arr = np.array(estimated_fluxes_ref[model])  # shape (N, 6)
    for flux_name in flux_keys:
        col = arr[:, col_index[flux_name]]
        rows.append({
            "model": model,
            "flux": flux_name,
            "true_value": true_flux_map[flux_name],
            "median": float(np.median(col)),
            "min": float(np.min(col)),
            "max": float(np.max(col)),
            "q25": float(np.quantile(col, 0.25)),
            "q75": float(np.quantile(col, 0.75)),
        })

summary_df_toy = pl.DataFrame(rows)
summary_df_toy.write_csv(RESULTS_DIR / "estimated_fluxes_reference_summary.csv")
print("Summary statistics for estimated fluxes in the Toy network:")
print(summary_df_toy.to_pandas().to_string(index=False))

Summary statistics for estimated fluxes in the Toy network:


model flux  true_value     median       min        max        q25        q75
 trad   v2       110.0 110.008046 73.651228 268.187925 101.215998 120.092465
 trad   v3        50.0  50.042676 25.596455 196.283764  43.539688  58.002219
 trad   v4        20.0  20.020106 14.047919  26.859560  18.894322  21.168993
 trad   v5        20.0  20.020106 14.047919  26.859560  18.894322  21.168993
 trad   v6        80.0  79.979894 73.140440  85.952081  78.831007  81.105678
  ilr   v2       110.0 110.063892 73.336159 267.988939 101.188921 120.049179
  ilr   v3        50.0  50.019449 25.029079 196.085066  43.512680  57.902435
  ilr   v4        20.0  20.013297 14.048063  26.756916  18.895274  21.155269
  ilr   v5        20.0  20.013297 14.048063  26.756916  18.895274  21.155269
  ilr   v6        80.0  79.986703 73.243084  85.951937  78.844731  81.104726


In [4]:
# # The following code produces LaTex tables for summary statistics
# col_rename = {
#     "flux": "Flux",
#     "true_value": "True value",
#     "median": "Median",
#     "min": "Min",
#     "max": "Max",
#     "q25": "Q25",
#     "q75": "Q75",
# }

# for model, label in (("trad", "Traditional I-MFA"), ("ilr", "Compositional I-MFA")):
#     df = (
#         summary_df_toy.filter(pl.col("model") == model)
#         .drop("model")
#         .rename(col_rename)
#         .to_pandas()
#     )
#     latex = df.to_latex(
#         index=False,
#         float_format="%.3f",
#         caption=f"Estimated fluxes — {label}",
#         label=f"tab:toy_fluxes_{model}",
#     )
#     print(f"% --- {label} ---")
#     print(latex)